# BreaQ 2026 — Challenge 1: Trust No Input

**Layer:** application and circuit construction  
**Difficulty:** beginner core, intermediate exploit, advanced extension  
**Recommended time:** 3–5 hours  
**Team roles:** investigator, exploit developer, defense analyst

> A BreaQ Cloud customer reports that a quantum policy-fingerprint service sometimes rejects valid assets after migration. Operations claims that the quantum circuit is unreliable. Your task is to determine whether the failure is quantum noise, incorrect circuit construction, or unsafe application-layer interpretation.

This challenge runs entirely on a local ideal simulator. No cloud account or real device is required.

## Rules and deliverables

All supplied local artifacts are in scope. Reading package source is allowed, but **finding a suspicious line is not sufficient evidence**. Your team must demonstrate the failure experimentally and provide a generalized mitigation.

Submit:

1. A threat model and one-sentence security claim.
2. A minimal reproducer.
3. Evidence localizing the fault.
4. A secure decoder or validation wrapper.
5. At least one negative test showing your defense does not merely hard-code the demonstration cases.
6. `submission.json`, generated near the end of this notebook.

External assistants and documentation may be used, but every member must be able to defend the submitted reasoning.

## Architecture under investigation

```text
Asset registry ── expected fingerprint
                         │
Manifest ──► circuit builder ──► ideal backend ──► raw counts
   │                                              │
   └──────── compatibility metadata ──► report adapter ──► MATCH / MISMATCH
```

### Contract

- Four logical slots are ordered as `alpha, beta, gamma, delta`.
- Slot `i` is encoded on Qiskit qubit `i`.
- The registry fingerprint is written in the canonical logical-slot order.
- Legacy compatibility metadata may affect presentation, but it must not silently change a security decision.

Your first job is to decide which data crosses a trust boundary and which representation is authoritative.

In [114]:
from pathlib import Path
import json
from itertools import permutations

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

from breaq_ch1 import ChallengeService

INSTANCE = Path("team_instance.json")
service = ChallengeService(INSTANCE)
print("Team:", service.team_id)
print("Cases:", [case.case_id for case in service.list_cases()])

Team: GravityFalls
Cases: ['CAL-ALPHA', 'CAL-BETA', 'MIG-DELTA', 'INCIDENT-7']


## Warm-up: Qiskit bit order

Qiskit labels qubits from `q0` upward, while displayed classical count strings place the highest-index classical bit on the left. That distinction is harmless when explicitly converted and dangerous when separate components silently assume different conventions.

Run the two-qubit example and explain why the visual circuit and count key are consistent.

In [115]:
warmup = QuantumCircuit(2, 2)
warmup.x(0)
warmup.measure([0, 1], [0, 1])
print(warmup.draw(output="text"))
print("Expected displayed key: 01")

     ┌───┐┌─┐
q_0: ┤ X ├┤M├
     └┬─┬┘└╥┘
q_1: ─┤M├──╫─
      └╥┘  ║ 
c: 2/══╩═══╩═
       1   0 
Expected displayed key: 01


**Warm-up question:** If the displayed count key is `0110`, what is the corresponding sequence in the canonical order `q0, q1, q2, q3`?

In [116]:
warmup_canonical = "0110"
warmup_explanation = ("Qiskit prints the highest classical bit on the left, so the key is q3 q2 q1 q0.\nReversing gives q0..q3 = 0110 (a palindrome, so it looks unchanged; that is key 0101 would become 1010).")
print("It outputs:", warmup_canonical)
print(warmup_explanation)

It outputs: 0110
Qiskit prints the highest classical bit on the left, so the key is q3 q2 q1 q0.
Reversing gives q0..q3 = 0110 (a palindrome, so it looks unchanged; that is key 0101 would become 1010).


## Phase A — Triage

Execute every case. Start with observations, not assumptions. Record the expected registry value, reported value, decision, dominant raw key, and provenance.

In [117]:
case_ids = [case.case_id for case in service.list_cases()]
results = {case_id: service.execute(case_id) for case_id in case_ids}

for case_id, result in results.items():
    print(f"{case_id:12} expected={result['registry_fingerprint']} "
          f"reported={result['reported_fingerprint']} "
          f"decision={result['decision']:8} counts={result['counts']}")

CAL-ALPHA    expected=1001 reported=1001 decision=MATCH    counts={'1001': 512}
CAL-BETA     expected=0110 reported=0110 decision=MATCH    counts={'0110': 512}
MIG-DELTA    expected=1010 reported=0110 decision=MISMATCH counts={'0101': 512}
INCIDENT-7   expected=1100 reported=0101 decision=MISMATCH counts={'0011': 512}


In [118]:
# Read the information from JSON table
evidence_table = []
for case_id, r in results.items():
    m = service.get_manifest(case_id)
    raw = max(r["counts"], key=r["counts"].get)
    evidence_table.append({
        "case_id": case_id,
        "registry": r["registry_fingerprint"],
        "raw_key": raw,
        "shots_on_raw_key": f"{r['counts'][raw]}/512",
        "canonical": raw[::-1],
        "reported": r["reported_fingerprint"],
        "decision": r["decision"],
        "compatibility_map": m["compatibility_map"],
        "provenance": m["provenance"],
    })
evidence_table

[{'case_id': 'CAL-ALPHA',
  'registry': '1001',
  'raw_key': '1001',
  'shots_on_raw_key': '512/512',
  'canonical': '1001',
  'reported': '1001',
  'decision': 'MATCH',
  'compatibility_map': [0, 1, 2, 3],
  'provenance': 'factory-calibration'},
 {'case_id': 'CAL-BETA',
  'registry': '0110',
  'raw_key': '0110',
  'shots_on_raw_key': '512/512',
  'canonical': '0110',
  'reported': '0110',
  'decision': 'MATCH',
  'compatibility_map': [0, 1, 2, 3],
  'provenance': 'factory-calibration'},
 {'case_id': 'MIG-DELTA',
  'registry': '1010',
  'raw_key': '0101',
  'shots_on_raw_key': '512/512',
  'canonical': '1010',
  'reported': '0110',
  'decision': 'MISMATCH',
  'compatibility_map': [1, 0, 2, 3],
  'provenance': 'migration-adapter'},
 {'case_id': 'INCIDENT-7',
  'registry': '1100',
  'raw_key': '0011',
  'shots_on_raw_key': '512/512',
  'canonical': '1100',
  'reported': '0101',
  'decision': 'MISMATCH',
  'compatibility_map': [3, 1, 2, 0],
  'provenance': 'external-import'}]

### Threat model

Complete this before inspecting implementation details.

| Element | Your answer |
|---|---|
| Protected asset | |
| Required property | |
| Adversary capability | |
| Trusted input | |
| Untrusted input | |
| Observable symptom | |
| Initial alternative explanations | |

In [119]:
threat_model = {
    "asset": "Integrity of the MATCH/MISMATCH authorization decision",
    "security_property": "The decision depends only on the measured (canonical) "
                         "fingerprint and the trusted registry value",
    "adversary_capability": "Can supply or edit an imported manifest, including its ",
    "trusted_inputs": ["registry fingerprint", "circuit builder", "simulator / raw counts"],
    "untrusted_inputs": ["manifest metadata: schema, compatibility_map, provenance"],
    "alternative_explanations": [
        "quantum noise",
        "wrong circuit (oracle encodes the wrong fingerprint)",
        "bit-order mistake when reading counts",
        "application layer reinterpreting the result",
    ],
}
threat_model

{'asset': 'Integrity of the MATCH/MISMATCH authorization decision',
 'security_property': 'The decision depends only on the measured (canonical) fingerprint and the trusted registry value',
 'adversary_capability': 'Can supply or edit an imported manifest, including its ',
 'trusted_inputs': ['registry fingerprint',
  'circuit builder',
  'simulator / raw counts'],
 'untrusted_inputs': ['manifest metadata: schema, compatibility_map, provenance'],
 'alternative_explanations': ['quantum noise',
  'wrong circuit (oracle encodes the wrong fingerprint)',
  'bit-order mistake when reading counts',
  'application layer reinterpreting the result']}

## Phase B — Localize the failure

For each incident, inspect three boundaries independently:

1. The circuit prepared for execution.
2. The raw backend output.
3. The application report.

Useful methods include drawing the circuit, using `Statevector` on a measurement-free copy, examining the manifest, and comparing equivalent representations. Do not assume that `reported_fingerprint` is raw quantum output.

In [120]:
focus_case = "MIG-DELTA"
manifest = service.get_manifest(focus_case)
circuit = service.get_circuit(focus_case)

print(json.dumps(manifest, indent=2))
print(circuit.draw(output="text"))

{
  "case_id": "MIG-DELTA",
  "schema": "legacy-0.9",
  "logical_slots": [
    "alpha",
    "beta",
    "gamma",
    "delta"
  ],
  "compatibility_map": [
    1,
    0,
    2,
    3
  ],
  "provenance": "migration-adapter"
}
     ┌───┐      ░            ░ ┌───┐┌─┐         
q_0: ┤ H ├──────░───■────────░─┤ H ├┤M├─────────
     ├───┤      ░   │        ░ ├───┤└╥┘┌─┐      
q_1: ┤ H ├──────░───┼────────░─┤ H ├─╫─┤M├──────
     ├───┤      ░   │        ░ ├───┤ ║ └╥┘┌─┐   
q_2: ┤ H ├──────░───┼────■───░─┤ H ├─╫──╫─┤M├───
     ├───┤      ░   │    │   ░ ├───┤ ║  ║ └╥┘┌─┐
q_3: ┤ H ├──────░───┼────┼───░─┤ H ├─╫──╫──╫─┤M├
     ├───┤┌───┐ ░ ┌─┴─┐┌─┴─┐ ░ └───┘ ║  ║  ║ └╥┘
q_4: ┤ X ├┤ H ├─░─┤ X ├┤ X ├─░───────╫──╫──╫──╫─
     └───┘└───┘ ░ └───┘└───┘ ░       ║  ║  ║  ║ 
c: 4/════════════════════════════════╩══╩══╩══╩═
                                     0  1  2  3 


In [ ]:
localization_evidence = {}
for case_id in case_ids:
    r = results[case_id]
    qc = service.get_circuit(case_id).remove_final_measurements(inplace=False)
    probs = Statevector(qc).probabilities_dict(qargs=range(4))
    sv_key = max(probs, key=probs.get)
    raw = max(r["counts"], key=r["counts"].get)
    localization_evidence[case_id] = {
        "registry": r["registry_fingerprint"],
        "1_circuit_statevector": f"{sv_key[::-1]} (P={probs[sv_key]:.1f})",
        "2_raw_counts_canonical": raw[::-1],
        "3_reported": r["reported_fingerprint"],
        "first_wrong_stage": next((s for s, v in [("circuit", sv_key[::-1]),
                                                   ("counts", raw[::-1]),
                                                   ("report", r["reported_fingerprint"])]
                                   if v != r["registry_fingerprint"]), "none"),
    }
localization_evidence

{'CAL-ALPHA': {'registry': '1001',
  '1_circuit_statevector': '1001 (P=1.0)',
  '2_raw_counts_canonical': '1001',
  '3_reported': '1001',
  'first_wrong_stage': 'none'},
 'CAL-BETA': {'registry': '0110',
  '1_circuit_statevector': '0110 (P=1.0)',
  '2_raw_counts_canonical': '0110',
  '3_reported': '0110',
  'first_wrong_stage': 'none'},
 'MIG-DELTA': {'registry': '1010',
  '1_circuit_statevector': '1010 (P=1.0)',
  '2_raw_counts_canonical': '1010',
  '3_reported': '0110',
  'first_wrong_stage': 'report'},
 'INCIDENT-7': {'registry': '1100',
  '1_circuit_statevector': '1100 (P=1.0)',
  '2_raw_counts_canonical': '1100',
  '3_reported': '0101',
  'first_wrong_stage': 'report'}}

### Your causal claim

State one falsifiable sentence, for example: *Changing X while holding Y and Z constant changes W.* Then demonstrate it in code.

In [122]:
causal_claim = ""
# TODO: controlled experiment supporting or refuting the claim.
controlled_experiment = {}
controlled_experiment

{}

## Phase C — Minimal reproducer

Produce the smallest example that changes the application result without changing the encoded quantum fingerprint. A good reproducer should have:

- one source fingerprint;
- two metadata configurations;
- identical canonical quantum results;
- different application reports.

In [123]:
# This is minimum because is the swap of two positions

def minimal_reproducer(service):
    fingerprint = "1010" #MIG-DELTA
    configs = {"identity": (0, 1, 2, 3), "legacy_swap": (1, 0, 2, 3)}
    runs = {name: service.simulate_adapter(fingerprint, circuit) for name, circuit in (configs.items())}
    return {
        "fingerprint": fingerprint,
        "runs": runs,
        "same_canonical": len({r["canonical_fingerprint"] for r in runs.values()}) == 1,
        "different_report": len({r["reported_fingerprint"] for r in runs.values()}) > 1,
    }

reproducer = minimal_reproducer(service)
reproducer

{'fingerprint': '1010',
 'runs': {'identity': {'counts': {'0101': 512},
   'canonical_fingerprint': '1010',
   'reported_fingerprint': '1010'},
  'legacy_swap': {'counts': {'0101': 512},
   'canonical_fingerprint': '1010',
   'reported_fingerprint': '0110'}},
 'same_canonical': True,
 'different_report': True}

## Phase D — Defensive fix

Implement a trusted decoder for security decisions. It must work for fingerprints not present in the demonstration instance.

**Requirement:** Legacy presentation may still exist, but the authorization comparison must use canonical logical-slot order.

In [ ]:
def secure_decode(counts: dict[str, int]) -> str:
    if not counts:
        raise ValueError("Empty count")
    key = max(counts, key = counts.get).replace(" ", " ")
    if set(key) - {"0", "1"}:
        raise ValueError("malformed count key")
    return key[::-1]


def secure_decision(counts: dict[str, int], registry_fingerprint: str) -> str:
    measured = secure_decode(counts)
    if len(registry_fingerprint) != len(measured) or set(registry_fingerprint) - {"0", "1"}:
        raise ValueError("malformed registry fingerprint")
    return "MATCH" if measured == registry_fingerprint else "MISMATCH"

In [125]:
# TODO: add positive and negative tests.
# Include fingerprints not present in team_instance.json.
def run_defense_tests():
    
    return {"implemented": False, "passed": 0, "failed": 0}

defense_results = run_defense_tests()
defense_results

{'implemented': False, 'passed': 0, 'failed': 0}

## Phase E — Exploit objective

Assume an attacker can modify only the compatibility permutation in an imported manifest. Determine when a chosen target report is reachable from a source fingerprint.

Your exploit must:

- return a valid permutation when the target is reachable;
- return `None` when it is impossible;
- verify the result through `service.simulate_adapter`;
- explain the limitation imposed by the number of zeroes and ones.

In [126]:
# Choose permutation
def craft_compatibility_map(source: str, target: str):
    # TODO: return a tuple permutation or None.
    if source.count("1") != target.count("1"):
        return None
    used = []
    for bit in target:
        for i in range(len(source)):
            if source[i] == bit and i not in used:
                used.append(i)
                break
    
    return tuple(used)

attack_demo = {}
for source, target in [("1010", "0110"), ("1010", "1100"), ("1010", "1110"), ("1000", "0000")]:
    m = craft_compatibility_map(source, target)
    reported = service.simulate_adapter(source, m)["reported_fingerprint"] if m else None
    attack_demo[f"{source} -> {target}"] = {"compatibility_map": m, "reported": reported, "success": reported == target }
attack_demo

{'1010 -> 0110': {'compatibility_map': (1, 0, 2, 3),
  'reported': '0110',
  'success': True},
 '1010 -> 1100': {'compatibility_map': (0, 2, 1, 3),
  'reported': '1100',
  'success': True},
 '1010 -> 1110': {'compatibility_map': None,
  'reported': None,
  'success': False},
 '1000 -> 0000': {'compatibility_map': None,
  'reported': None,
  'success': False}}

## Research extension

Choose at least one:

- Exhaustively test all 4-bit fingerprints and all 24 permutations; quantify how many permutations alter each fingerprint.
- Design a manifest-validation policy that distinguishes security semantics from presentation semantics.
- Add an authenticated binding for security-relevant metadata and explain what it does **not** protect.
- Generalize the decoder and exploit to `n` bits, then discuss scalability.
- Construct a metamorphic test suite that detects order-dependent application behavior without knowing the secret fingerprint in advance.

In [127]:
from itertools import product, permutations
fingerprints = [''.join(bits) for bits in product('01', repeat=4)]
position_permutations = permutations(range(4))
for fingerprint in fingerprints:
    changed = sum(''.join(fingerprint[i] for i in perm) != fingerprint for perm in permutations(range(4)))
    print(f"{fingerprint}: {changed} permutations change it")

0000: 0 permutations change it
0001: 18 permutations change it
0010: 18 permutations change it
0011: 20 permutations change it
0100: 18 permutations change it
0101: 20 permutations change it
0110: 20 permutations change it
0111: 18 permutations change it
1000: 18 permutations change it
1001: 20 permutations change it
1010: 20 permutations change it
1011: 18 permutations change it
1100: 20 permutations change it
1101: 18 permutations change it
1110: 18 permutations change it
1111: 0 permutations change it


## Claim sheet

Complete this table before preparing the presentation.

| Claim | Evidence cell/output | Assumption | Limitation |
|---|---|---|---|
| The fault is / is not quantum noise | | | |
| The circuit does / does not encode the registry value | | | |
| Metadata can / cannot alter the decision | | | |
| The mitigation generalizes | | | |

In [128]:
claim_sheet = [
    {
        "claim": "The fault is not quantum noise; it is caused by application-layer interpretation of the result.",
        "evidence": "Phase B circuit inspection and Phase C controlled experiment showing whether the same encoded fingerprint produces different reports under different compatibility metadata.",
        "assumption": "The local simulator is ideal and the encoded circuit is unchanged between the compared runs.",
        "limitation": "This conclusion applies to the tested local challenge cases; it does not rule out noise on real quantum hardware."
    },
    {
        "claim": "The circuit encodes the registry fingerprint, but the raw Qiskit count key must be converted to canonical logical-slot order before comparison.",
        "evidence": "Compare the registry fingerprint, dominant raw count key, reversed canonical key, and reported fingerprint in the Phase A evidence table and Phase B circuit inspection.",
        "assumption": "Qubit i represents logical slot i, as specified by the challenge contract.",
        "limitation": "A matching bit order alone does not prove that every possible circuit or measurement mapping is correct."
    },
    {
        "claim": "Compatibility metadata can alter the reported fingerprint and potentially affect the MATCH/MISMATCH decision.",
        "evidence": "Phase C minimal reproducer: hold the encoded fingerprint fixed, vary only the compatibility metadata, and compare the resulting reports and decisions.",
        "assumption": "The two experiments differ only in the compatibility metadata being tested.",
        "limitation": "The demonstrated effect depends on the adapter's behavior and the tested metadata configurations."
    },
    {
        "claim": "The mitigation generalizes because it decodes raw measurement counts into canonical order and compares that result with the registry, independently of compatibility metadata.",
        "evidence": "Phase D positive and negative tests, including fingerprints not present in team_instance.json and tests using different bit patterns.",
        "assumption": "The measurement mapping and canonical logical-slot order are trusted and explicitly defined.",
        "limitation": "The decoder must also validate malformed counts, ties, invalid fingerprints, and any assumptions about shot distributions before being used in production."
    }
]

claim_sheet

[{'claim': 'The fault is not quantum noise; it is caused by application-layer interpretation of the result.',
  'evidence': 'Phase B circuit inspection and Phase C controlled experiment showing whether the same encoded fingerprint produces different reports under different compatibility metadata.',
  'assumption': 'The local simulator is ideal and the encoded circuit is unchanged between the compared runs.',
  'limitation': 'This conclusion applies to the tested local challenge cases; it does not rule out noise on real quantum hardware.'},
 {'claim': 'The circuit encodes the registry fingerprint, but the raw Qiskit count key must be converted to canonical logical-slot order before comparison.',
  'evidence': 'Compare the registry fingerprint, dominant raw count key, reversed canonical key, and reported fingerprint in the Phase A evidence table and Phase B circuit inspection.',
  'assumption': 'Qubit i represents logical slot i, as specified by the challenge contract.',
  'limitation': 

## Export submission

Fill the fields below. The exported JSON is not a substitute for the completed notebook; it gives the jury a machine-readable index of your claims.

In [129]:
submission = {
    "team_id": service.team_id,
    "challenge": "01-trust-no-input",
    "security_claim": "An attacker who can edit only the untrusted compatibility_map in an imported manifest can force a MATCH or MISMATCH result against any registry fingerprint with the same number of 1-bits, without changing the circuit, backend or registry, because the adapter compares the registry value with a reordered copy of the result instead of the measured q₀…q₃ fingerprint.",
    "root_cause_layer": "",
    "minimal_reproducer": reproducer,
    "defense_results": defense_results,
    "attack_demo": attack_demo,
    "claims": claim_sheet,
    "team_member_contributions": ["Pascu Mihai-Daniel", "Oprea Rares-Mihai", "Brebenel Mihnea-Stefan"],
}

Path("submission.json").write_text(json.dumps(submission, indent=2), encoding="utf-8")
print("Wrote submission.json")

Wrote submission.json


## Prepare for oral defense

Every member should be ready to answer at least one of these:

- Why is the dominant Qiskit count key not automatically the canonical logical fingerprint?
- Which controlled experiment excludes circuit noise as the root cause?
- Can every chosen target be produced by permutation alone? Why?
- Why is validating that a map is a permutation insufficient?
- Where should legacy presentation conversion occur?
- What happens if the jury gives you a fresh fingerprint and compatibility map?

### Technical references

- IBM Quantum, *Bit ordering in Qiskit*: https://quantum.cloud.ibm.com/docs/en/guides/bit-ordering
- IBM Quantum, `Statevector.probabilities_dict`: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector
- Qiskit BasicSimulator API: https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.providers.basic_provider.BasicSimulator